In [ ]:
import os
import sys

os.environ["TQDM_DISABLE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

import tqdm.auto
import tqdm.std

tqdm.auto.tqdm = tqdm.std.tqdm
tqdm.auto.trange = tqdm.std.trange

import logging
from pathlib import Path

from drzavni_posao import (
    CharacterAssignerConfig,
    EmbedderConfig,
    MediaStore,
    MlflowTracker,
    PipelineConfig,
    Stage,
    TranscriberConfig,
    run_pipeline,
)
from drzavni_posao.gallery import load_speaker_samples

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)

logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("mlflow.system_metrics").setLevel(logging.WARNING)

logger = logging.getLogger("drzavni_posao.pipeline")

In [ ]:
EXPERIMENT_NAME = (
    "/Users/luka.erceg@databricks.com/drzavni_posao/transcription"
)
AUDIO_PATHS = Path("/Volumes/luka_erceg/drzavni_posao/audios")
TRANSCRIPT_PATH = Path(
    "/Workspace/Users/luka.erceg@databricks.com/drzavni_posao/transcripts"
)
GALLERY_PATH = Path(
    "/Workspace/Users/luka.erceg@databricks.com/drzavni_posao/data/speaker_galleries/"
    "gallery_extended.json"
)

HF_TOKEN = dbutils.secrets.get(scope="luka-erceg", key="hf_token")

DEVICE = "cuda"
COMPUTE_TYPE = "bfloat16"
ASR_MODEL = "large-v3"
ALIGN_MODEL = "classla/wav2vec2-xls-r-parlaspeech-hr"
EMBEDDER_MODEL = "pyannote/wespeaker-voxceleb-resnet34-LM"
MIN_SEGMENT_SECONDS = 0.025
VAD_REALIGN_MIN_SILENCE = 1.0
VAD_REALIGN_PADDING = 0.1

CAST_HOTWORDS = (
    "Đorđe Čvarkov, Dragan Torbica, Boškić, Dobrosavljev, "
    "Marijanović, Živan Radosavljević, Frau Šilovička"
)
ASR_OPTIONS = {
    "no_repeat_ngram_size": 5,
    "repetition_penalty": 1.0,
    "hotwords": CAST_HOTWORDS,
}

tracker = MlflowTracker(EXPERIMENT_NAME)

In [ ]:
SPEAKER_SAMPLES = load_speaker_samples(GALLERY_PATH)

In [ ]:
media = MediaStore(
    AUDIO_PATHS,
    TRANSCRIPT_PATH,
    trim_start=25,
    trim_end=7,
)

character_assigner_config = CharacterAssignerConfig(
    min_similarity=0.1,
    embedder=EmbedderConfig(
        device=DEVICE,
        hf_token=HF_TOKEN,
        embed_model=EMBEDDER_MODEL,
        min_segment_seconds=MIN_SEGMENT_SECONDS,
    ),
    speaker_samples=SPEAKER_SAMPLES,
)

config = PipelineConfig.from_media(
    media,
    transcription=TranscriberConfig(
        device=DEVICE,
        compute_type=COMPUTE_TYPE,
        model_name=ASR_MODEL,
        language="sr",
        align_model_name=ALIGN_MODEL,
        asr_options=ASR_OPTIONS,
        vad_repair_min_silence=VAD_REALIGN_MIN_SILENCE,
        vad_repair_padding=VAD_REALIGN_PADDING,
    ),
    character_assigner=character_assigner_config,
)

ALL_EPISODES = {episode.key for episode in media.episodes()}
len(ALL_EPISODES), config.generated_name()

In [ ]:
RUN_ID = run_pipeline(
    tracker,
    media,
    config,
    ALL_EPISODES,
    through=Stage.CHARACTER_ASSIGNER,
    tags={"experiment": "vad-realignment", "gallery": "extended"},
)
RUN_ID